**Precision-Optimized Credit Card Fraud Detection Pipeline**

_Leveraging Machine Learning to Capture Marginal Fraud and Protect Solvency_

Facundo Urquizo - Statistics and Quantitative Modeling - Data Science Track (Baruch College - Zicklin School of Business)

In [2]:
#Importing methods and libraries
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import pandas as pd
import zipfile
#Unziping file to open CSV
with zipfile.ZipFile('creditcard.zip', 'r') as z:
    creditcard_file = pd.read_csv(z.open('creditcard.csv'))
classifier = RandomForestClassifier()
creditcard_file.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [4]:
#Extracted normal and fraudulent transactions from the dataset and assigned them to variables
fraud_credit_card = creditcard_file[creditcard_file['Class'] == 1]
normal_credit_card = creditcard_file[creditcard_file['Class'] == 0]

X = creditcard_file.drop(columns=['Class'])
y = creditcard_file['Class']
X_train, X_test, y_train, Y_test =train_test_split(X, y, test_size = 0.2, random_state = 2)
#Standard Scaler: Equal mathematical weight
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
#SMOTE (Synthetic Minority Over-sampling Technique): Generate  artificial fraud
smote = SMOTE(random_state=4)
model = LogisticRegression(max_iter=5000)
X_train_smote, y_train_smote  = smote.fit_resample(X_train_scaled, y_train)
#Random Forest Classifier: Evaluate Transactions
classifier = RandomForestClassifier(random_state=42)
classifier.fit(X_train_smote, y_train_smote)
y_pred = classifier.predict(X_test_scaled)
print(classification_report(Y_test,y_pred))
probabilities = classifier.predict_proba(X_test_scaled)[:, 1]

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56878
           1       0.88      0.83      0.85        84

    accuracy                           1.00     56962
   macro avg       0.94      0.92      0.93     56962
weighted avg       1.00      1.00      1.00     56962



Threshold Tuning to further navigate the Precision-Recall Trade-Off

In [4]:
#Strengthening precision from 88 to 92 by requiring (75%+) algorithmic consensus
probabilities = classifier.predict_proba(X_test_scaled)[:, 1]
strict_guesses = (probabilities >= 0.75).astype(int)
print("Strict Threshold (75%) Scorecard:")
print(classification_report(Y_test, strict_guesses))

Strict Threshold (75%) Scorecard:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56878
           1       0.92      0.82      0.87        84

    accuracy                           1.00     56962
   macro avg       0.96      0.91      0.93     56962
weighted avg       1.00      1.00      1.00     56962

